# Spatially varying diffusion coefficients

It is possible to create spatially varying diffusion coefficient input files for emc3_eirene

In [ ]:
import xarray as xr
import xemc3
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Use local helper function to get some data
from get_data import load_example_data

ds = load_example_data()
# If you want to use your own data use something like
# ds = xemc3.load.all("path/to/mydata/")
# or if you have converted it already to a netcdf file
# ds = xr.open_dataset("path/to/mydata.nc")

In [ ]:
vol_dims = [x[0] for x in [ds.r.shape, ds.theta.shape, ds.phi.shape]]
print(vol_dims)

## Create the desired profiles

In [ ]:
phi = np.linspace(0, np.pi / 5, vol_dims[2])[None, :]
theta = np.linspace(0, np.pi * 2, vol_dims[1])[:, None]
data = np.exp(-(((phi) * 4) ** 2) - (theta - np.pi * 1) ** 2)

## Plot what you made

In [ ]:
plt.pcolormesh(data)

## Store in emc3_eirene format

In [ ]:
for base in 1, 10:
    d2 = 0.1 + (base - 1) * 0.1 * data
    k = "diff"
    ds[k] = ds.Te
    ds[k].attrs = {}
    ds[k].data[...] = d2 * 1e4
    prefix = f"pol_tor_v1_base={base}_"
    ds.emc3.to_fort(k, f"{prefix}_diffusion.fort", kinetic=True)
    ds[k].data *= 3
    ds.emc3.to_fort(k, f"{prefix}_heat.fort", kinetic=True)